# Khởi tạo Spark Session
Kết nối với MinIO để đọc dữ liệu Delta tables từ S3. Yêu cầu đã cài đặt `pyspark`, `delta-spark`.

In [1]:
import delta
from pyspark.sql import SparkSession

extra_packages = ["org.apache.hadoop:hadoop-aws:3.4.2", "software.amazon.awssdk:bundle:2.29.52"]

# Khởi tạo Spark Session với cấu hình MinIO & Delta Lake (Hỗ trợ PySpark 4.1.x)
builder = (
    SparkSession.builder.appName("DeltaTableExplorer")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:9000")
    .config("spark.hadoop.fs.s3a.access.key", "minio")
    .config("spark.hadoop.fs.s3a.secret.key", "minio123456")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
)

# BẮT BUỘC dùng tham số extra_packages, nếu không hàm này sẽ ghi đè đè mất hadoop-aws
spark = delta.configure_spark_with_delta_pip(builder, extra_packages=extra_packages).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print("Spark Session initiated!")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/11 12:58:31 WARN Utils: Your hostname, cuong-Nitro-AN515-56, resolves to a loopback address: 127.0.1.1; using 192.168.1.125 instead (on interface wlp0s20f3)
26/05/11 12:58:31 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/cuong/Desktop/DATN/source/.venv/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/cuong/.ivy2.5.2/cache
The jars for the packages stored in: /home/cuong/.ivy2.5.2/jars
io.delta#delta-spark_4.1_2.13 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
software.amazon.awssdk#bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-ae89e0a6-3ef3-46fa-9614-65c31ca559a9;1.0
	confs: [default]
	found io.delta#delta-spark_4.1_2.13;4.1.0 in central
	found io.delta#delta-storage;4.1.0 in

Spark Session initiated!


## Khám phá bảng Bronze (Dữ liệu thô)

In [2]:
df_bronze = spark.read.format("delta").load("s3a://lakehouse/mooc/bronze/mooc_events_raw")

print("=== SCHEMA: BRONZE ===")
df_bronze.printSchema()

print("\n=== MỘT VÀI DÒNG DỮ LIỆU ===")
df_bronze.select("kafka_topic", "ingest_ts", "value_raw").show(5, truncate=100)

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


=== SCHEMA: BRONZE ===
root
 |-- kafka_topic: string (nullable = true)
 |-- kafka_partition: integer (nullable = true)
 |-- kafka_offset: string (nullable = true)
 |-- kafka_timestamp: timestamp (nullable = true)
 |-- kafka_key: string (nullable = true)
 |-- value_raw: string (nullable = true)
 |-- ingest_ts: timestamp (nullable = true)
 |-- ingest_date: date (nullable = true)
 |-- ingest_hour: integer (nullable = true)
 |-- dedup_key: string (nullable = true)
 |-- parse_status: string (nullable = true)
 |-- parse_error: string (nullable = true)


=== MỘT VÀI DÒNG DỮ LIỆU ===


+---------------+-----------------------+----------------------------------------------------------------------------------------------------+
|    kafka_topic|              ingest_ts|                                                                                           value_raw|
+---------------+-----------------------+----------------------------------------------------------------------------------------------------+
|mooc.raw.events|2026-05-11 12:22:13.968|{"name": "/courses/course-v1:SoICT+IT4593+2023-1-Chien/xblock/block-v1:SoICT+IT4593+2023-1-Chien+...|
|mooc.raw.events|2026-05-11 12:22:13.968|{"name": "problem_check", "context": {"user_id": 62932, "path": "/event", "course_id": "course-v1...|
|mooc.raw.events|2026-05-11 12:22:13.968|{"name": "edx.grades.problem.submitted", "context": {"course_id": "course-v1:SoICT+IT4593+2023-1-...|
|mooc.raw.events|2026-05-11 12:22:13.968|{"name": "problem_check", "context": {"course_id": "course-v1:SoICT+IT4593+2023-1-Chien", "course...|

## Khám phá bảng Silver: Learning Events

In [3]:
df_learning = spark.read.format("delta").load("s3a://lakehouse/mooc/silver/learning_events")

print("=== SCHEMA: LEARNING EVENTS ===")
df_learning.printSchema()

print("\n=== MỘT VÀI DÒNG DỮ LIỆU ===")
df_learning.show(5, truncate=False)

=== SCHEMA: LEARNING EVENTS ===
root
 |-- event_id: string (nullable = true)
 |-- ts: timestamp (nullable = true)
 |-- event_date: date (nullable = true)
 |-- event_hour: integer (nullable = true)
 |-- event_type: string (nullable = true)
 |-- event_source: string (nullable = true)
 |-- username: string (nullable = true)
 |-- user_id: long (nullable = true)
 |-- session_id: string (nullable = true)
 |-- course_id: string (nullable = true)
 |-- org_id: string (nullable = true)
 |-- block_type: string (nullable = true)
 |-- block_id: string (nullable = true)
 |-- completion_value: integer (nullable = true)
 |-- ip: string (nullable = true)
 |-- host: string (nullable = true)
 |-- agent: string (nullable = true)
 |-- referer: string (nullable = true)
 |-- page: string (nullable = true)
 |-- event_json: string (nullable = true)


=== MỘT VÀI DÒNG DỮ LIỆU ===


+----------------------------------------------------------------+--------------------------+----------+----------+---------------------------------------------------------------------------------------------------------------------------------------------------------+------------+---------+-------+--------------------------------+------------------------------+-------+----------+--------------------------------+----------------+------------+---------------+-----------------------------------------------------------------------------------------------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------

## Khám phá bảng Silver: Video Interactions

In [6]:
df_video = spark.read.format("delta").load("s3a://lakehouse/mooc/silver/video_interactions")

print("=== SCHEMA: VIDEO INTERACTIONS ===")
df_video.printSchema()

print("\n=== MỘT VÀI DÒNG DỮ LIỆU ===")
df_video.show(10, truncate=False)

=== SCHEMA: VIDEO INTERACTIONS ===
root
 |-- event_id: string (nullable = true)
 |-- ts: timestamp (nullable = true)
 |-- event_date: date (nullable = true)
 |-- event_type: string (nullable = true)
 |-- username: string (nullable = true)
 |-- user_id: long (nullable = true)
 |-- session_id: string (nullable = true)
 |-- course_id: string (nullable = true)
 |-- org_id: string (nullable = true)
 |-- video_id: string (nullable = true)
 |-- video_code: string (nullable = true)
 |-- video_duration: double (nullable = true)
 |-- current_time: double (nullable = true)
 |-- old_time: double (nullable = true)
 |-- new_time: double (nullable = true)
 |-- seek_type: string (nullable = true)
 |-- old_speed: double (nullable = true)
 |-- new_speed: double (nullable = true)
 |-- saved_position: string (nullable = true)


=== MỘT VÀI DÒNG DỮ LIỆU ===
+----------------------------------------------------------------+--------------------------+----------+-----------------------------------------------

## Thống kê số lượng bản ghi của các bảng Silver

In [5]:
tables = [
    "learning_events",
    "performance_events",
    "exam_attempts",
    "video_interactions",
    "navigation_events",
    "pdf_interactions",
    "system_events",
    "unknown_events",
]

for t in tables:
    try:
        count = spark.read.format("delta").load(f"s3a://lakehouse/mooc/silver/{t}").count()
        print(f"Table {t}: {count} records")
    except Exception:
        print(f"Table {t} chưa có dữ liệu hoặc không tồn tại.")

Table learning_events: 3643 records


Table performance_events: 962 records


Table exam_attempts: 24 records
Table video_interactions: 217 records


Table navigation_events: 1191 records


Table pdf_interactions: 80 records


Table system_events: 240 records


Table unknown_events: 0 records


## Test quá trình OPTIMIZE Delta Table
Đếm số lượng file thực tế của 1 partition TRƯỚC và SAU khi chạy lệnh OPTIMIZE.

In [ ]:
from delta.tables import DeltaTable

bronze_path = "s3a://lakehouse/mooc/bronze/mooc_events_raw"
partition_condition = "ingest_date='2026-05-11' and ingest_hour=4"

# 1. Lấy danh sách file TRƯỚC khi optimize
df_partition = spark.read.format("delta").load(bronze_path).filter(partition_condition)
files_before = df_partition.inputFiles()
print(f"Số lượng file TRƯỚC khi Optimize: {len(files_before)}")

# 2. Chạy OPTIMIZE
print("Đang chạy OPTIMIZE...")
spark.sql(f"OPTIMIZE delta.`{bronze_path}` WHERE {partition_condition}").show(truncate=False)

# 3. Lấy danh sách file SAU khi optimize
# Phải load lại dataframe để lấy snapshot mới nhất của Delta
df_partition_after = spark.read.format("delta").load(bronze_path).filter(partition_condition)
files_after = df_partition_after.inputFiles()
print(f"Số lượng file SAU khi Optimize: {len(files_after)}")

# 4. Xem History ghi nhận thao tác OPTIMIZE
dt = DeltaTable.forPath(spark, bronze_path)
dt.history(1).select(
    "version",
    "timestamp",
    "operation",
    "operationMetrics.numFilesAdded",
    "operationMetrics.numFilesRemoved",
).show(truncate=False)